<a href="https://colab.research.google.com/github/SamManuJacob/6thSem-ML-Lab/blob/main/1BM23CS291_Lab6_DECISION_TREES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ===================== IMPORTS =====================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder


# ===================== IRIS DATASET =====================
iris = pd.read_csv('iris (1).csv')

X = iris.drop('species', axis=1)
y = iris['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("========== IRIS DATASET ==========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


# ===================== DRUG DATASET =====================
drug = pd.read_csv('drug.csv')

le = LabelEncoder()
for col in drug.columns:
    if drug[col].dtype == object:
        drug[col] = le.fit_transform(drug[col])

X = drug.drop('Drug', axis=1)
y = drug['Drug']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n========== DRUG DATASET ==========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


# ===================== PETROL DATASET =====================
petrol = pd.read_csv('petrol_consumption.csv')

X = petrol.drop('Petrol_Consumption', axis=1)
y = petrol['Petrol_Consumption']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = DecisionTreeRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n========== PETROL DATASET ==========")
print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)

========== IRIS DATASET ==========
Accuracy: 1.0
Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

========== DRUG DATASET ==========
Accuracy: 1.0
Confusion Matrix:
 [[ 6  0  0  0  0]
 [ 0  3  0  0  0]
 [ 0  0  5  0  0]
 [ 0  0  0 11  0]
 [ 0  0  0  0 15]]

========== PETROL DATASET ==========
Mean Absolute Error: 83.7
Mean Squared Error: 15552.5
Root Mean Squared Error: 124.70966281728133


In [11]:
  # ===================== IMPORTS =====================
import pandas as pd
import numpy as np
from collections import Counter


# ===================== HELPER FUNCTIONS =====================

def entropy(y):
    counts = Counter(y)
    total = len(y)
    ent = 0
    for c in counts.values():
        p = c / total
        ent -= p * np.log2(p)
    return ent


def variance(y):
    return np.var(y)


def split_dataset(X, y, feature, threshold):
    left_idx = X[feature] <= threshold
    right_idx = X[feature] > threshold
    return X[left_idx], X[right_idx], y[left_idx], y[right_idx]


def best_split_classification(X, y):
    best_gain = -1
    best_feature, best_thresh = None, None
    base_entropy = entropy(y)

    for feature in X.columns:
        thresholds = X[feature].unique()
        for t in thresholds:
            X_l, X_r, y_l, y_r = split_dataset(X, y, feature, t)

            if len(y_l) == 0 or len(y_r) == 0:
                continue

            p_l = len(y_l) / len(y)
            p_r = len(y_r) / len(y)

            gain = base_entropy - (p_l * entropy(y_l) + p_r * entropy(y_r))

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_thresh = t

    return best_feature, best_thresh


def best_split_regression(X, y):
    best_var = float('inf')
    best_feature, best_thresh = None, None

    for feature in X.columns:
        thresholds = X[feature].unique()
        for t in thresholds:
            X_l, X_r, y_l, y_r = split_dataset(X, y, feature, t)

            if len(y_l) == 0 or len(y_r) == 0:
                continue

            var = (len(y_l) * variance(y_l) + len(y_r) * variance(y_r)) / len(y)

            if var < best_var:
                best_var = var
                best_feature = feature
                best_thresh = t

    return best_feature, best_thresh


# ===================== TREE BUILD =====================

def build_tree_classification(X, y, depth=0, max_depth=3):
    if len(set(y)) == 1 or depth == max_depth:
        return Counter(y).most_common(1)[0][0]

    feature, thresh = best_split_classification(X, y)

    if feature is None:
        return Counter(y).most_common(1)[0][0]

    X_l, X_r, y_l, y_r = split_dataset(X, y, feature, thresh)

    return {
        'feature': feature,
        'threshold': thresh,
        'left': build_tree_classification(X_l, y_l, depth+1, max_depth),
        'right': build_tree_classification(X_r, y_r, depth+1, max_depth)
    }


def build_tree_regression(X, y, depth=0, max_depth=3):
    if len(y) <= 2 or depth == max_depth:
        return np.mean(y)

    feature, thresh = best_split_regression(X, y)

    if feature is None:
        return np.mean(y)

    X_l, X_r, y_l, y_r = split_dataset(X, y, feature, thresh)

    return {
        'feature': feature,
        'threshold': thresh,
        'left': build_tree_regression(X_l, y_l, depth+1, max_depth),
        'right': build_tree_regression(X_r, y_r, depth+1, max_depth)
    }


# ===================== PREDICTION =====================

def predict(tree, row):
    if not isinstance(tree, dict):
        return tree

    if row[tree['feature']] <= tree['threshold']:
        return predict(tree['left'], row)
    else:
        return predict(tree['right'], row)


# ===================== METRICS =====================

def accuracy(y_true, y_pred):
    return sum(y_true == y_pred) / len(y_true)


def confusion_matrix_calc(y_true, y_pred):
    classes = sorted(list(set(list(y_true) + list(y_pred))))

    matrix = np.zeros((len(classes), len(classes)), dtype=int)

    for i in range(len(y_true)):
        actual = classes.index(y_true[i])
        pred = classes.index(y_pred[i])
        matrix[actual][pred] += 1

    print("Classes:", classes)
    return matrix


# ===================== IRIS =====================
iris = pd.read_csv('iris (1).csv')

X = iris.drop('species', axis=1)
y = iris['species']

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split].values, y[split:].values

tree = build_tree_classification(X_train, y_train)

y_pred = [predict(tree, X_test.iloc[i]) for i in range(len(X_test))]

print("========== IRIS ==========")
print("Accuracy:", accuracy(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix_calc(y_test, y_pred))


# ===================== DRUG =====================
drug = pd.read_csv('drug.csv')

# Encode manually
for col in drug.columns:
    if drug[col].dtype == object:
        drug[col] = pd.factorize(drug[col])[0]

X = drug.drop('Drug', axis=1)
y = drug['Drug']

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split].values, y[split:].values

tree = build_tree_classification(X_train, y_train)

y_pred = [predict(tree, X_test.iloc[i]) for i in range(len(X_test))]

print("\n========== DRUG ==========")
print("Accuracy:", accuracy(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix_calc(y_test, y_pred))


# ===================== PETROL =====================
petrol = pd.read_csv('petrol_consumption.csv')

X = petrol.drop('Petrol_Consumption', axis=1)
y = petrol['Petrol_Consumption']

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split].values, y[split:].values

tree = build_tree_regression(X_train, y_train)

y_pred = np.array([predict(tree, X_test.iloc[i]) for i in range(len(X_test))])

mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)

print("\n========== PETROL ==========")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

========== IRIS ==========
Accuracy: 0.7333333333333333
Classes: ['versicolor', 'virginica']
Confusion Matrix:
 [[ 0  0]
 [ 8 22]]

========== DRUG ==========
Accuracy: 0.925
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Confusion Matrix:
 [[20  0  0  0  0]
 [ 0  0  3  0  0]
 [ 0  0 10  0  0]
 [ 0  0  0  5  0]
 [ 0  0  0  0  2]]

========== PETROL ==========
MAE: 83.84285714285714
MSE: 15755.136734693873
RMSE: 125.51946755262259
